In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [ ]:
comparison_dataset_train = load_dataset(path="openai/summarize_from_feedback", name="comparisons", split="train").map(lambda x: {"id": x["info"]["id"]})
comparison_dataset_val = load_dataset(path="openai/summarize_from_feedback", name="comparisons", split="validation").map(lambda x: {"id": x["info"]["id"]})

In [ ]:
comparison_dataset_train.num_rows, comparison_dataset_val.num_rows

In [ ]:
sft_dataset_train = pd.read_json("datasets/sft/train.jsonl", lines=True)
sft_dataset_val = pd.read_json("datasets/sft/valid.jsonl", lines=True)
sft_dataset_test = pd.read_json("datasets/sft/test.jsonl", lines=True)

sft_dataset_train.shape[0], sft_dataset_val.shape[0], sft_dataset_test.shape[0]

Here we ensure there are no data leaks between subsets:

In [ ]:
sft_dataset = pd.concat([sft_dataset_train, sft_dataset_val, sft_dataset_test], ignore_index=True)
sft_distinct_ids = sft_dataset["id"]
assert sft_distinct_ids.shape[0] == sft_distinct_ids.unique().shape[0]

In [ ]:
assert set(comparison_dataset_train["id"]) & set(comparison_dataset_val["id"]) == set()

In [ ]:
comparison_ids = set(comparison_dataset_train["id"]) | set(comparison_dataset_val["id"])
comparison_sft_intersection_train = set(sft_distinct_ids) & set(comparison_dataset_train["id"])
comparison_sft_intersection_val = set(sft_distinct_ids) & set(comparison_dataset_val["id"])
len(comparison_sft_intersection_train), len(comparison_sft_intersection_val)

In [ ]:
(
    sft_dataset_train.loc[~sft_dataset_train["id"].isin(comparison_sft_intersection_train | comparison_sft_intersection_val)].shape[0],
    sft_dataset_val.loc[~sft_dataset_val["id"].isin(comparison_sft_intersection_train | comparison_sft_intersection_val)].shape[0],
    sft_dataset_test.loc[~sft_dataset_test["id"].isin(comparison_sft_intersection_train | comparison_sft_intersection_val)].shape[0]
)

Prevent sft dataset from intersecting with RM data:

In [ ]:
sft_dataset_dedup =sft_dataset.loc[
    ~sft_dataset["id"].isin(comparison_sft_intersection_train | comparison_sft_intersection_val)
].reset_index(drop=True)

In [ ]:
SEED = 228
n_total = sft_dataset_dedup.shape[0]

sft_dataset_dedup_train, sft_dataset_dedup_test = train_test_split(
    sft_dataset_dedup, test_size=.2, random_state=SEED
)
sft_dataset_dedup_train, sft_dataset_dedup_val = train_test_split(
    sft_dataset_dedup_train, test_size=1/8, random_state=SEED
)
sft_dataset_dedup_train.shape[0] / n_total, sft_dataset_dedup_val.shape[0] / n_total, sft_dataset_dedup_test.shape[0] / n_total

In [ ]:
sft_dataset_dedup_train.reset_index(drop=True).to_parquet("datasets/sft/train_dedup.parquet")
sft_dataset_dedup_val.reset_index(drop=True).to_parquet("datasets/sft/val_dedup.parquet")
sft_dataset_dedup_test.reset_index(drop=True).to_parquet("datasets/sft/test_dedup.parquet")

In [ ]:
comparison_dataset_train.save_to_disk("datasets/comparison/train")
comparison_dataset_val.save_to_disk("datasets/comparison/val")